## Research Question:
Which team performance metrics most strongly predict upsets in the NCAA tournament?

For this question, we will use team performance metrics located in the online Bart Torvik dataset from 2023 to 2025.

The following metrics will be considered: AdjOE, AdjDE, Barthag, EFG%, EFGD%, TOR, TORD, ORB, DRB, FTR, FTRD, 2P%, 2P%D, 3P%, 3P%D, 3PR, 3PRD, Adj_T, WAB.

Definitions:
AdjOE = Adjusted Offensive Efficiency,
AdjDE = Adjusted Defense Efficiency,
Barthag = Power Rating (Chance of beating average D1 team),
EFG% = Effective Field Goal Percentage,
EFGD% = Average Opposing Team Effective Field Goal Percentage,
TOR = Turnover Percentage,
TORD = Average Opposing Team Turnover Percentage,
ORB = Rebounding Percentage,
DRB = Average Opposing Team Rebounding Percentage,
FTR = Free-Throw Rate,
FTRD = Average Opposing Team Free-Throw Rate,
2P% = Two Point Percentage,
2P%D = Average Opposing Team Two Point Percentage,
3P% = Three Point Percentage,
3P%D = Average Opposing Team Three Point Percentage,
3PR = Three Point Rate,
3PRD = Average Opposing Team Three Point Rate,
Adj_T = Adjusted Tempo,
WAB = Wins Above Bubble

In [1]:
import pandas as pd

def get_torvik_team_stats(year):
# Reads the locally stored Barttorvik CSV for the given year and assigns column headers
    url = f"torvik_{year}.csv"

    try:
        df = pd.read_csv(url, header=None, skiprows=1)

        full_headers = [
            'Team', 'AdjOE', 'AdjDE', 'Barthag', 'Record', 'Wins', 'G',
            'EFG%', 'EFGD%', 'FTR', 'FTRD', 'TOR', 'TORD', 'ORB', 'DRB',
            'Adj_T', '2P%', '2P%D', '3P%', '3P%D', 'Col21', 'Col22',
            'Col23', 'Col24', '3PR', '3PRD', 'Col27', 'Col28', 'Col29', 'Col30',
            'Season', 'Col32', 'Col33', 'Col34', 'WAB', 'Col36', 'Col37'
        ]

        df.columns = full_headers[:len(df.columns)]
        df['Season'] = year
        df = df[df['Team'] != 'Team'].reset_index(drop=True)
        df = df.drop(columns=[c for c in ['RK', 'Conf', 'G', 'Postseason'] if c in df.columns])
        return df

    except Exception as e:
        print(f"Error {year}: {e}")
        return None


In [2]:
tourney_results = pd.read_csv('MNCAATourneyCompactResults.csv')
tourney_seeds = pd.read_csv('MNCAATourneySeeds.csv')
teams = pd.read_csv('MTeams.csv')

tourney_seeds['SeedInt'] = tourney_seeds['Seed'].apply(lambda x: int(''.join(filter(str.isdigit, x))))

# Merge winner and loser seeds onto tournament results
tourney_results = tourney_results.merge(
    tourney_seeds[['Season', 'TeamID', 'SeedInt']],
    left_on=['Season', 'WTeamID'],
    right_on=['Season', 'TeamID'],
    how='left'
).rename(columns={'SeedInt': 'WSeed'}).drop('TeamID', axis=1)

tourney_results = tourney_results.merge(
    tourney_seeds[['Season', 'TeamID', 'SeedInt']],
    left_on=['Season', 'LTeamID'],
    right_on=['Season', 'TeamID'],
    how='left'
).rename(columns={'SeedInt': 'LSeed'}).drop('TeamID', axis=1)

tourney_results['SeedDiff'] = (tourney_results['WSeed'] - tourney_results['LSeed']).abs()

# Filter to upset opportunities (seed diff >= 5, 2023-2025)
opportunities = tourney_results[
    (tourney_results['SeedDiff'] >= 5) &
    (tourney_results['Season'] >= 2023) &
    (tourney_results['Season'] <= 2025)
].copy()

# split into upsets that happened (upset successes) and upsets that did not happen (upset failures)
upset_successes = opportunities[opportunities['WSeed'] > opportunities['LSeed']].reset_index(drop=True)
upset_fails = opportunities[opportunities['WSeed'] < opportunities['LSeed']].reset_index(drop=True)

In [3]:
import numpy as np

# Name mapping dictionary to match Barttorvik team names with Kaggle dataset team names
name_map = {
    "Saint Mary's": "St Mary's CA", "San Diego St.": "San Diego St", "Florida Atlantic": "FL Atlantic",
    "Kansas St.": "Kansas St", "Michigan St.": "Michigan St", "Iowa St.": "Iowa St",
    "Utah St.": "Utah St", "Penn St.": "Penn St", "Oklahoma St.": "Oklahoma St",
    "Boise St.": "Boise St", "Ohio St.": "Ohio St", "N.C. State": "NC State",
    "Arizona St.": "Arizona St", "Washington St.": "Washington St", "Mississippi St.": "Mississippi St",
    "Charleston": "Col Charleston", "Sam Houston St.": "Sam Houston St", "St. John's": "St John's",
    "Kent St.": "Kent", "Wichita St.": "Wichita St", "Indiana St.": "Indiana St",
    "Saint Louis": "St Louis", "Montana St.": "Montana St", "San Jose St.": "San Jose St",
    "Colorado St.": "Colorado St", "Kennesaw St.": "Kennesaw", "Cal St. Fullerton": "CS Fullerton",
    "UMass Lowell": "MA Lowell", "Middle Tennessee": "MTSU", "Tarleton St.": "Tarleton St",
    "Loyola Marymount": "Loy Marymount", "Eastern Washington": "E Washington", "Fresno St.": "Fresno St",
    "Stephen F. Austin": "SF Austin", "Southern Illinois": "S Illinois", "Missouri St.": "Missouri St",
    "Eastern Kentucky": "E Kentucky", "Northern Kentucky": "N Kentucky", "Appalachian St.": "Appalachian St",
    "Long Beach St.": "Long Beach St", "Youngstown St.": "Youngstown St", "North Carolina Central": "NC Central",
    "Cleveland St.": "Cleveland St", "Ball St.": "Ball St", "Grambling St.": "Grambling",
    "Saint Joseph's": "St Joseph's PA", "Western Kentucky": "WKU", "St. Bonaventure": "St Bonaventure",
    "Texas A&M Corpus Chris": "TAM C. Christi", "Georgia Southern": "Ga Southern", "Texas St.": "Texas St",
    "St. Thomas": "St Thomas MN", "George Washington": "G Washington", "Wright St.": "Wright St",
    "Abilene Christian": "Abilene Chr", "FIU": "Florida Intl", "Weber St.": "Weber St",
    "South Dakota St.": "S Dakota St", "Florida Gulf Coast": "FGCU", "Queens": "Queens NC",
    "Florida St.": "Florida St", "New Mexico St.": "New Mexico St", "Morehead St.": "Morehead St",
    "Oregon St.": "Oregon St", "Detroit Mercy": "Detroit", "Norfolk St.": "Norfolk St",
    "Murray St.": "Murray St", "Northwestern St.": "Northwestern LA", "Loyola Chicago": "Loyola-Chicago",
    "Jacksonville St.": "Jacksonville St", "Northern Colorado": "N Colorado", "North Dakota St.": "N Dakota St",
    "Portland St.": "Portland St", "Milwaukee": "WI Milwaukee", "Purdue Fort Wayne": "PFW",
    "Western Carolina": "W Carolina", "Northern Illinois": "N Illinois", "Idaho St.": "Idaho St",
    "USC Upstate": "SC Upstate", "East Tennessee St.": "ETSU", "Mount St. Mary's": "Mt St Mary's",
    "Sacramento St.": "CS Sacramento", "UTSA": "UT San Antonio", "UT Rio Grande Valley": "UTRGV",
    "Alcorn St.": "Alcorn St", "Southeastern Louisiana": "SE Louisiana", "SIU Edwardsville": "SIUE",
    "Fairleigh Dickinson": "F Dickinson", "Southeast Missouri St.": "SE Missouri St", "Maryland Eastern Shore": "MD E Shore",
    "Louisiana Monroe": "ULM", "Prairie View A&M": "Prairie View", "Nicholls St.": "Nicholls St",
    "Tennessee St.": "Tennessee St", "Tennessee Martin": "TN Martin", "American": "American Univ",
    "Illinois Chicago": "IL Chicago", "Arkansas St.": "Arkansas St", "Coastal Carolina": "Coastal Car",
    "Texas Southern": "TX Southern", "Boston University": "Boston Univ", "Georgia St.": "Georgia St",
    "Illinois St.": "Illinois St", "Cal St. Bakersfield": "CS Bakersfield", "Charleston Southern": "Charleston So",
    "Southern": "Southern Univ", "Western Illinois": "W Illinois", "Chicago St.": "Chicago St",
    "Jackson St.": "Jackson St", "Saint Peter's": "St Peter's", "Eastern Michigan": "E Michigan",
    "Arkansas Pine Bluff": "Ark Pine Bluff", "Nebraska Omaha": "NE Omaha", "Morgan St.": "Morgan St",
    "North Carolina A&T": "NC A&T", "Saint Francis": "St Francis PA", "Little Rock": "Ark Little Rock",
    "Central Arkansas": "Cent Arkansas", "Cal St. Northridge": "CS Northridge", "McNeese St.": "McNeese St",
    "UMKC": "Missouri KC", "Western Michigan": "W Michigan", "The Citadel": "Citadel",
    "Coppin St.": "Coppin St", "South Carolina St.": "S Carolina St", "Central Michigan": "C Michigan",
    "Central Connecticut": "Central Conn", "Albany": "SUNY Albany", "Bethune Cookman": "Bethune-Cookman",
    "Monmouth": "Monmouth NJ", "Houston Christian": "Houston Chr", "Eastern Illinois": "E Illinois",
    "St. Francis NY": "St Francis NY", "Alabama St.": "Alabama St", "Mississippi Valley St.": "MS Valley St",
    "Delaware St.": "Delaware St", "IU Indy": "IUPUI", "Green Bay": "WI Green Bay", "LIU": "LIU Brooklyn"
}

tourney_seeds['SeedInt'] = tourney_seeds['Seed'].apply(lambda x: int(''.join(filter(str.isdigit, x))))
annual_analysis_frames = []

for year in range(2023, 2026):
    torvik_year = get_torvik_team_stats(year)
    if torvik_year is None: continue

    year_results = tourney_results[tourney_results['Season'] == year].copy()
    year_results = year_results.drop(columns=['WSeed', 'LSeed', 'SeedDiff'], errors='ignore')

    seed_map = tourney_seeds[tourney_seeds['Season'] == year][['TeamID', 'SeedInt']].drop_duplicates(subset=['TeamID'])

    year_results = year_results.merge(seed_map, left_on='WTeamID', right_on='TeamID', how='left').rename(columns={'SeedInt': 'WSeed'})
    year_results = year_results.drop(columns=['TeamID'], errors='ignore')

    year_results = year_results.merge(seed_map, left_on='LTeamID', right_on='TeamID', how='left').rename(columns={'SeedInt': 'LSeed'})
    year_results = year_results.drop(columns=['TeamID'], errors='ignore')

    w_seeds = year_results['WSeed']
    l_seeds = year_results['LSeed']
    if isinstance(w_seeds, pd.DataFrame): w_seeds = w_seeds.iloc[:, 0]
    if isinstance(l_seeds, pd.DataFrame): l_seeds = l_seeds.iloc[:, 0]

    year_results['SeedDiff'] = (w_seeds - l_seeds).abs()

    opportunities = year_results[year_results['SeedDiff'] >= 5].copy()
    opportunities['Upset_Occurred'] = (opportunities['WSeed'] > opportunities['LSeed']).astype(int)

    torvik_year = torvik_year.dropna(subset=['Team'])
    torvik_year['MappedName'] = torvik_year['Team'].astype(str).replace(name_map)

    torvik_year = torvik_year.merge(teams[['TeamID', 'TeamName']], left_on='MappedName', right_on='TeamName', how='left').dropna(subset=['TeamID'])
    torvik_year['TeamID'] = torvik_year['TeamID'].astype(int)

    stats_cols = ['AdjOE', 'AdjDE', 'Barthag', 'EFG%', 'EFGD%', 'TOR', 'TORD', 'ORB', 'DRB', 'FTR',
                  'FTRD', '2P%', '2P%D', '3P%', '3P%D', '3PR', '3PRD', 'Adj_T', 'WAB']

    torvik_stats = torvik_year[['TeamID'] + stats_cols].drop_duplicates(subset=['TeamID']).copy()

    for col in stats_cols:
        torvik_stats[col] = pd.to_numeric(
            torvik_stats[col].astype(str).str.replace('%', '').str.strip(),
            errors='coerce'
        )

    overlap = set(torvik_stats['TeamID']) & set(opportunities['WTeamID'])

    opportunities['WTeamID'] = opportunities['WTeamID'].astype(int)
    opportunities['LTeamID'] = opportunities['LTeamID'].astype(int)

# Merge winner and loser stats onto each game
    opportunities = opportunities.merge(torvik_stats, left_on='WTeamID', right_on='TeamID', how='left').drop(columns=['TeamID'], errors='ignore')
    opportunities = opportunities.rename(columns={c: f"{c}_W" for c in stats_cols})

    opportunities = opportunities.merge(torvik_stats, left_on='LTeamID', right_on='TeamID', how='left').drop(columns=['TeamID'], errors='ignore')
    opportunities = opportunities.rename(columns={c: f"{c}_L" for c in stats_cols})

    annual_analysis_frames.append(opportunities)

# Combine all years, compute underdog minus favorite stat differences, and correlate with upset outcomes
master_study = pd.concat(annual_analysis_frames, ignore_index=True).copy()

diff_dict = {}
for m in stats_cols:
    w_col, l_col = f'{m}_W', f'{m}_L'
    if w_col in master_study.columns and l_col in master_study.columns:
        w_vals = master_study[w_col].values
        l_vals = master_study[l_col].values
        w_seeds = master_study['WSeed'].values
        l_seeds = master_study['LSeed'].values

        underdog_vals = np.where(w_seeds > l_seeds, w_vals, l_vals)
        favorite_vals = np.where(w_seeds > l_seeds, l_vals, w_vals)
        diff_dict[f'{m}_Diff'] = underdog_vals - favorite_vals

diff_df = pd.DataFrame(diff_dict)
diff_df['Upset_Occurred'] = master_study['Upset_Occurred'].values


correlations = diff_df.dropna().corr()
print(f"Games Analyzed: {len(diff_df.dropna())}")
print(correlations['Upset_Occurred'].sort_values(ascending=False))
  

Games Analyzed: 113
Upset_Occurred    1.000000
AdjOE_Diff        0.269314
Barthag_Diff      0.256196
WAB_Diff          0.235249
ORB_Diff          0.071737
EFG%_Diff         0.050940
2P%_Diff          0.049227
Adj_T_Diff        0.033759
3P%_Diff          0.030443
FTR_Diff          0.029486
3PR_Diff          0.011294
3PRD_Diff        -0.075418
FTRD_Diff        -0.083148
2P%D_Diff        -0.105253
TOR_Diff         -0.126045
TORD_Diff        -0.146325
3P%D_Diff        -0.153556
DRB_Diff         -0.154618
EFGD%_Diff       -0.156344
AdjDE_Diff       -0.232665
Name: Upset_Occurred, dtype: float64


This analysis examined 113 tournament Matchups from 2023 to 2025 where the seed difference was at least five, comparing underdog and favorite team statistics from Bart Torvik data to determine which metrics best predict upset occurrences. The results reveal the Pearson correlation coefficient (r) associated with each metric for upset probability. The magnitude of r indicates the strength of the relationship. The sign reflects the nature of each metric — offensive metrics are positive because higher values indicate superiority, while defensive metrics are negative because lower values indicate better defense.

On the offensive side, we see that AdjOE and Barthag are the strongest predictors of upsets, displaying r values of 0.269 and 0.256 respectively. This emphasizes the importance of offensive output for underdogs. Additionally, WAB is also a strong predictor (r = 0.235), suggesting that underdogs who have performed well against strong competition during the regular season perform better in the tournament.

Defensively, AdjDE (r = -0.233), EFGD% (r = -0.156), and 3P%D (r = -0.154) showed strong correlations. Therefore, underdogs that defend the perimeter effectively relative to their opponent are more likely to pull off upsets. 


    